# Multilingual LVC dataset

## Spanish

Version in current use

### acitve sentences

In [ ]:
import os
import pandas as pd

# =========================
# OUTPUT DIRECTORY
# =========================

output_dir = "ES_active_passive"
os.makedirs(output_dir, exist_ok=True)

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_ES_v1_utf8.csv"
file2 = "sentence_buildersES_utf8.csv"
file3 = "spanish_verbs_inflection_sub_utf8_agreement.csv"

# =========================
# READ FILES
# =========================

items = pd.read_csv(
    file1,
    sep=None,
    engine="python",
    encoding="utf8"
)

builder = pd.read_csv(
    file2,
    sep=None,
    engine="python",
    encoding="utf8"
)

verbs = pd.read_csv(
    file3,
    sep=None,
    engine="python",
    encoding="utf8"
)

# =========================
# CLEAN DATA
# =========================

for df in [items, builder, verbs]:

    # Clean column names
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    # Remove accidental unnamed columns
    df.drop(
        columns=[
            c for c in df.columns
            if c.startswith("Unnamed")
        ],
        inplace=True
    )

    # Strip whitespace without turning NaN into "nan"
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(
                lambda x: x.strip()
                if isinstance(x, str)
                else x
            )

# Remove rows without a usable verb
items = items[
    items["verb_es"].notna()
    & (items["verb_es"] != "")
].copy()

# =========================
# SENTENCE BUILDER ELEMENTS
# =========================

time_specs = (
    builder["time_specs"]
    .dropna()
)

time_specs = (
    time_specs[
        time_specs != ""
    ]
    .unique()
    .tolist()
)

adverbs = (
    builder["adverbs"]
    .dropna()
)

adverbs = (
    adverbs[
        adverbs != ""
    ]
    .unique()
    .tolist()
)

# 3rd person plural lookup only;
# pronoun itself is omitted from the sentence
subjects = ["ellos"]

# =========================
# TENSE COLUMNS
# =========================

tense_columns = {
    "future": "future_active",
    "present_perfect": "present_perfect_active",
    "past_perfect": "past_perfect_active",
}

# =========================
# OUTPUT CONTAINERS
# =========================

output_rows = {
    "future_no_adverbs": [],
    "future_with_adverbs": [],
    "present_perfect_no_adverbs": [],
    "present_perfect_with_adverbs": [],
    "past_perfect_no_adverbs": [],
    "past_perfect_with_adverbs": [],
}

# =========================
# HELPER FUNCTIONS
# =========================

def clean_missing(x):
    """
    Convert missing values to empty string
    and strip whitespace from strings.
    """
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.lower() == "nan":
        return ""

    return x


def get_object_phrase(item):
    """
    Use full_expression as the primary reference.

    Example:
        verb_es = dar
        full_expression = dar un consejo

    returns:
        un consejo

    Falls back to object_es if full_expression
    is missing or unusable.
    """

    verb = clean_missing(
        item.get("verb_es", "")
    )

    full_expression = clean_missing(
        item.get("full_expression", "")
    )

    object_es = clean_missing(
        item.get("object_es", "")
    )

    # Normal case:
    # "dar un consejo" -> "un consejo"
    if (
        full_expression
        and verb
        and full_expression.startswith(verb + " ")
    ):
        return full_expression[
            len(verb) + 1:
        ].strip()

    # Fallback:
    # remove first word of full_expression
    if full_expression:
        parts = full_expression.split(" ", 1)

        if len(parts) == 2:
            return parts[1].strip()

    # Final fallback
    return object_es


def get_verb_form(
    verb,
    subject,
    tense_col
):

    row = verbs[
        (verbs["verb"] == verb)
        & (verbs["person"] == subject)
    ]

    if row.empty:
        raise ValueError(
            f"No verb form found for "
            f"verb={verb}, "
            f"subject={subject}, "
            f"tense={tense_col}"
        )

    verb_form = row.iloc[0][tense_col]

    if pd.isna(verb_form):
        raise ValueError(
            f"Missing inflected form for "
            f"verb={verb}, "
            f"subject={subject}, "
            f"tense={tense_col}"
        )

    return str(verb_form).strip()


def add_adverb(
    verb_form,
    adverb,
    tense
):
    """
    Future:
        darán finalmente

    Compound tense:
        han finalmente dado
        habían finalmente dado
    """

    if tense == "future":
        return (
            f"{verb_form} {adverb}"
        )

    aux, participle = verb_form.split(
        " ",
        1
    )

    return (
        f"{aux} {adverb} {participle}"
    )


# =========================
# GENERATE SENTENCES
# =========================

for _, item in items.iterrows():

    verb = clean_missing(
        item["verb_es"]
    )

    obj = get_object_phrase(
        item
    )

    # Skip unusable rows
    if not verb or not obj:
        continue

    for time_spec in time_specs:

        for subject in subjects:

            for tense, tense_col in tense_columns.items():

                verb_form = get_verb_form(
                    verb,
                    subject,
                    tense_col
                )

                # =====================
                # WITHOUT ADVERB
                # =====================

                sentence = (
                    f"{time_spec}, "
                    f"{verb_form} {obj}."
                )

                row = item.to_dict()

                row.update({
                    "sentence": sentence,
                    "voice": "active",
                    "tense": tense,
                    "adverb": "NA",
                    "subject": subject,
                    "subject_realized": "no",
                    "time_spec": time_spec
                })

                output_rows[
                    f"{tense}_no_adverbs"
                ].append(row)

                # =====================
                # WITH ADVERB
                # =====================

                for adverb in adverbs:

                    verb_form_adv = add_adverb(
                        verb_form,
                        adverb,
                        tense
                    )

                    sentence = (
                        f"{time_spec}, "
                        f"{verb_form_adv} {obj}."
                    )

                    row = item.to_dict()

                    row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "subject": subject,
                        "subject_realized": "no",
                        "time_spec": time_spec
                    })

                    output_rows[
                        f"{tense}_with_adverbs"
                    ].append(row)

# =========================
# SAVE OUTPUT FILES
# =========================

for name, rows in output_rows.items():

    out = pd.DataFrame(
        rows
    )

    # Remove completely empty columns
    out = out.dropna(
        axis=1,
        how="all"
    )

    filename = os.path.join(
        output_dir,
        f"LVC_ES_active_{name}.csv"
    )

    out.to_csv(
        filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved {len(out)} rows "
        f"to {filename}"
    )

### passive

In [ ]:
import os
import pandas as pd

# =========================
# OUTPUT DIRECTORY
# =========================

output_dir = "ES_active_passive"
os.makedirs(output_dir, exist_ok=True)

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_ES_v1_utf8.csv"
file2 = "sentence_buildersES_utf8.csv"
file3 = "spanish_verbs_inflection_sub_utf8_agreement.csv"

# =========================
# READ FILES
# =========================

items = pd.read_csv(
    file1,
    sep=None,
    engine="python",
    encoding="utf8"
)

builder = pd.read_csv(
    file2,
    sep=None,
    engine="python",
    encoding="utf8"
)

verbs = pd.read_csv(
    file3,
    sep=None,
    engine="python",
    encoding="utf8"
)

# =========================
# CLEAN DATA
# =========================

for df in [items, builder, verbs]:

    # Clean column names
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    # Remove accidental unnamed columns
    df.drop(
        columns=[
            c for c in df.columns
            if c.startswith("Unnamed")
        ],
        inplace=True
    )

    # Strip whitespace without converting NaN to "nan"
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(
                lambda x: x.strip()
                if isinstance(x, str)
                else x
            )

# Remove rows without a usable verb
items = items[
    items["verb_es"].notna()
    & (items["verb_es"] != "")
].copy()

# =========================
# SENTENCE BUILDER ELEMENTS
# =========================

time_specs = (
    builder["time_specs"]
    .dropna()
)

time_specs = (
    time_specs[
        time_specs != ""
    ]
    .unique()
    .tolist()
)

adverbs = (
    builder["adverbs"]
    .dropna()
)

adverbs = (
    adverbs[
        adverbs != ""
    ]
    .unique()
    .tolist()
)

# =========================
# TENSE COLUMNS
# =========================

tense_columns = {
    "future": "future_passive",
    "present_perfect": "present_perfect_passive",
    "past_perfect": "past_perfect_passive",
}

# =========================
# OUTPUT CONTAINERS
# =========================

output_rows = {
    "future_no_adverbs": [],
    "future_with_adverbs": [],
    "present_perfect_no_adverbs": [],
    "present_perfect_with_adverbs": [],
    "past_perfect_no_adverbs": [],
    "past_perfect_with_adverbs": [],
}

# =========================
# HELPER FUNCTIONS
# =========================

def clean_missing(x):
    """
    Convert missing values to empty strings
    and strip whitespace.
    """
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.lower() == "nan":
        return ""

    return x


def get_object_phrase(item):
    """
    Use full_expression as the primary source
    for the passive subject phrase.

    Example:
        verb_es = dar
        full_expression = dar un consejo

    returns:
        un consejo

    Example:
        verb_es = dar
        full_expression = dar ayuda

    returns:
        ayuda

    Falls back to object_es if necessary.
    """

    verb = clean_missing(
        item.get("verb_es", "")
    )

    full_expression = clean_missing(
        item.get("full_expression", "")
    )

    object_es = clean_missing(
        item.get("object_es", "")
    )

    # Expected case:
    # "dar un consejo" -> "un consejo"
    # "dar ayuda" -> "ayuda"
    if (
        full_expression
        and verb
        and full_expression.startswith(verb + " ")
    ):
        return full_expression[
            len(verb) + 1:
        ].strip()

    # Fallback: remove first word
    if full_expression:
        parts = full_expression.split(" ", 1)

        if len(parts) == 2:
            return parts[1].strip()

    # Final fallback
    return object_es


def get_passive_person(gender, number, obj):
    """
    Determine which row of the inflection table
    should be used from the explicit gender and
    number annotations.

    M + sg -> él
    F + sg -> ella
    M + pl -> ellos
    F + pl -> ellas
    """

    gender = clean_missing(gender).upper()
    number = clean_missing(number).lower()

    if gender == "M" and number == "sg":
        return "él"

    if gender == "F" and number == "sg":
        return "ella"

    if gender == "M" and number == "pl":
        return "ellos"

    if gender == "F" and number == "pl":
        return "ellas"

    raise ValueError(
        f"Unexpected gender/number combination: "
        f"object={obj}, "
        f"gender={gender}, "
        f"number={number}"
    )


def get_verb_form(
    verb,
    passive_person,
    tense_col
):
    """
    Retrieve the already gender- and number-correct
    passive form from the corrected inflection table.
    """

    row = verbs[
        (verbs["verb"] == verb)
        & (verbs["person"] == passive_person)
    ]

    if row.empty:
        raise ValueError(
            f"No verb form found for "
            f"verb={verb}, "
            f"person={passive_person}, "
            f"tense={tense_col}"
        )

    verb_form = row.iloc[0][tense_col]

    if pd.isna(verb_form):
        raise ValueError(
            f"Missing passive form for "
            f"verb={verb}, "
            f"person={passive_person}, "
            f"tense={tense_col}"
        )

    return str(verb_form).strip()


def add_adverb(
    passive_form,
    adverb
):
    """
    Insert the adverb after the first auxiliary.

    Examples:

        será dada
        -> será finalmente dada

        ha sido dada
        -> ha finalmente sido dada

        habían sido dadas
        -> habían finalmente sido dadas
    """

    parts = passive_form.split(" ", 1)

    if len(parts) != 2:
        return f"{passive_form} {adverb}"

    aux, rest = parts

    return f"{aux} {adverb} {rest}"


# =========================
# GENERATE SENTENCES
# =========================

for _, item in items.iterrows():

    verb = clean_missing(
        item["verb_es"]
    )

    obj = get_object_phrase(
        item
    )

    gender = clean_missing(
        item["gen_object_es"]
    )

    number = clean_missing(
        item["num_obj_es"]
    )

    # Skip rows without usable verb/object
    if not verb or not obj:
        continue

    # Determine agreement row from explicit annotations
    passive_person = get_passive_person(
        gender,
        number,
        obj
    )

    for time_spec in time_specs:

        for tense, tense_col in tense_columns.items():

            verb_form = get_verb_form(
                verb,
                passive_person,
                tense_col
            )

            # =====================
            # WITHOUT ADVERB
            # =====================

            sentence = (
                f"{time_spec}, "
                f"{obj} {verb_form}."
            )

            row = item.to_dict()

            row.update({
                "sentence": sentence,
                "voice": "passive",
                "tense": tense,
                "adverb": "NA",
                "subject": obj,
                "passive_person": passive_person,
                "object_gender": gender,
                "object_number": number,
                "time_spec": time_spec
            })

            output_rows[
                f"{tense}_no_adverbs"
            ].append(row)

            # =====================
            # WITH ADVERB
            # =====================

            for adverb in adverbs:

                verb_form_adv = add_adverb(
                    verb_form,
                    adverb
                )

                sentence = (
                    f"{time_spec}, "
                    f"{obj} {verb_form_adv}."
                )

                row = item.to_dict()

                row.update({
                    "sentence": sentence,
                    "voice": "passive",
                    "tense": tense,
                    "adverb": adverb,
                    "subject": obj,
                    "passive_person": passive_person,
                    "object_gender": gender,
                    "object_number": number,
                    "time_spec": time_spec
                })

                output_rows[
                    f"{tense}_with_adverbs"
                ].append(row)

# =========================
# SAVE OUTPUT FILES
# =========================

for name, rows in output_rows.items():

    out = pd.DataFrame(rows)

    # Remove completely empty columns
    out = out.dropna(
        axis=1,
        how="all"
    )

    filename = os.path.join(
        output_dir,
        f"LVC_ES_passive_{name}.csv"
    )

    out.to_csv(
        filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved {len(out)} rows "
        f"to {filename}"
    )

In [ ]:
import os
import pandas as pd

# =========================
# OUTPUT DIRECTORY
# =========================

output_dir = "ES_active_passive"
os.makedirs(output_dir, exist_ok=True)

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_ES_v1_utf8.csv"
file2 = "sentence_buildersES_utf8.csv"
file3 = "spanish_verbs_inflection_sub_utf8.csv"

# =========================
# READ FILES
# =========================

items = pd.read_csv(
    file1,
    sep=None,
    engine="python",
    encoding="utf8"
)

builder = pd.read_csv(
    file2,
    sep=None,
    engine="python",
    encoding="utf8"
)

verbs = pd.read_csv(
    file3,
    sep=None,
    engine="python",
    encoding="utf8"
)

# =========================
# CLEAN DATA
# =========================

for df in [items, builder, verbs]:

    # Clean column names
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    # Remove accidental unnamed columns
    df.drop(
        columns=[
            c for c in df.columns
            if c.startswith("Unnamed")
        ],
        inplace=True
    )

    # Strip whitespace without turning NaN into "nan"
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(
                lambda x: x.strip()
                if isinstance(x, str)
                else x
            )

# Remove rows without a usable Spanish verb
items = items[
    items["verb_es"].notna()
    & (items["verb_es"] != "")
].copy()

# =========================
# SENTENCE BUILDER ELEMENTS
# =========================

time_specs = (
    builder["time_specs"]
    .dropna()
)

time_specs = (
    time_specs[
        time_specs != ""
    ]
    .unique()
    .tolist()
)

adverbs = (
    builder["adverbs"]
    .dropna()
)

adverbs = (
    adverbs[
        adverbs != ""
    ]
    .unique()
    .tolist()
)

# =========================
# TENSE COLUMNS
# =========================

tense_columns = {
    "future": "future_passive",
    "present_perfect": "present_perfect_passive",
    "past_perfect": "past_perfect_passive",
}

# =========================
# OUTPUT CONTAINERS
# =========================

output_rows = {
    "future_no_adverbs": [],
    "future_with_adverbs": [],
    "present_perfect_no_adverbs": [],
    "present_perfect_with_adverbs": [],
    "past_perfect_no_adverbs": [],
    "past_perfect_with_adverbs": [],
}

# =========================
# HELPER FUNCTIONS
# =========================

def clean_missing(x):
    """
    Convert missing values to empty strings
    and strip whitespace.
    """
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.lower() == "nan":
        return ""

    return x


def get_object_phrase(item):
    """
    Use full_expression as the primary source
    for the passive subject phrase.

    Example:
        verb_es = dar
        full_expression = dar un consejo

    returns:
        un consejo

    Falls back to object_es if necessary.
    """

    verb = clean_missing(
        item.get("verb_es", "")
    )

    full_expression = clean_missing(
        item.get("full_expression", "")
    )

    object_es = clean_missing(
        item.get("object_es", "")
    )

    # Normal case:
    # "dar un consejo" -> "un consejo"
    if (
        full_expression
        and verb
        and full_expression.startswith(verb + " ")
    ):
        return full_expression[
            len(verb) + 1:
        ].strip()

    # Fallback:
    # remove the first word from full_expression
    if full_expression:
        parts = full_expression.split(" ", 1)

        if len(parts) == 2:
            return parts[1].strip()

    # Final fallback
    return object_es


def infer_passive_person(obj):
    """
    Infer Spanish passive agreement from
    the determiner in the full object phrase.

    Examples:
        un consejo   -> él
        una decisión -> ella
        los consejos -> ellos
        las medidas  -> ellas
    """

    obj = clean_missing(obj).lower()

    if obj.startswith(("las ", "unas ")):
        return "ellas"

    if obj.startswith(("los ", "unos ")):
        return "ellos"

    if obj.startswith(("la ", "una ")):
        return "ella"

    if obj.startswith(("el ", "un ")):
        return "él"

    # Fallback if no determiner is available
    return "él"


def get_verb_form(
    verb,
    passive_person,
    tense_col
):

    row = verbs[
        (verbs["verb"] == verb)
        & (verbs["person"] == passive_person)
    ]

    if row.empty:
        raise ValueError(
            f"No verb form found for "
            f"verb={verb}, "
            f"person={passive_person}, "
            f"tense={tense_col}"
        )

    verb_form = row.iloc[0][tense_col]

    if pd.isna(verb_form):
        raise ValueError(
            f"Missing passive form for "
            f"verb={verb}, "
            f"person={passive_person}, "
            f"tense={tense_col}"
        )

    return str(verb_form).strip()


def add_adverb(
    passive_form,
    adverb,
    tense
):
    """
    Insert adverb after the auxiliary.

    Examples:

        será finalmente dado
        ha finalmente sido dado
        había finalmente sido dado
    """

    aux, rest = passive_form.split(
        " ",
        1
    )

    return (
        f"{aux} {adverb} {rest}"
    )


# =========================
# GENERATE SENTENCES
# =========================

for _, item in items.iterrows():

    verb = clean_missing(
        item["verb_es"]
    )

    # IMPORTANT:
    # derived primarily from full_expression
    obj = get_object_phrase(
        item
    )

    # Skip unusable rows
    if not verb or not obj:
        continue

    passive_person = infer_passive_person(
        obj
    )

    for time_spec in time_specs:

        for tense, tense_col in tense_columns.items():

            verb_form = get_verb_form(
                verb,
                passive_person,
                tense_col
            )

            # =====================
            # WITHOUT ADVERB
            # =====================

            sentence = (
                f"{time_spec}, "
                f"{obj} {verb_form}."
            )

            row = item.to_dict()

            row.update({
                "sentence": sentence,
                "voice": "passive",
                "tense": tense,
                "adverb": "NA",
                "subject": obj,
                "passive_person": passive_person,
                "time_spec": time_spec
            })

            output_rows[
                f"{tense}_no_adverbs"
            ].append(row)

            # =====================
            # WITH ADVERB
            # =====================

            for adverb in adverbs:

                verb_form_adv = add_adverb(
                    verb_form,
                    adverb,
                    tense
                )

                sentence = (
                    f"{time_spec}, "
                    f"{obj} {verb_form_adv}."
                )

                row = item.to_dict()

                row.update({
                    "sentence": sentence,
                    "voice": "passive",
                    "tense": tense,
                    "adverb": adverb,
                    "subject": obj,
                    "passive_person": passive_person,
                    "time_spec": time_spec
                })

                output_rows[
                    f"{tense}_with_adverbs"
                ].append(row)

# =========================
# SAVE OUTPUT FILES
# =========================

for name, rows in output_rows.items():

    out = pd.DataFrame(
        rows
    )

    # Remove completely empty columns
    out = out.dropna(
        axis=1,
        how="all"
    )

    filename = os.path.join(
        output_dir,
        f"LVC_ES_passive_{name}.csv"
    )

    out.to_csv(
        filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved {len(out)} rows "
        f"to {filename}"
    )

### nominal subjects

In [ ]:
import pandas as pd
import random
import os

# =========================
# SETTINGS
# =========================

SUBJECT_NUMBER = "both"      # "sg", "pl", "both"
SUBJECT_DET_TYPES = ["det"]  # "det", "det_poss", "no_det"

USE_PRONOUN_SUBJECTS = False
USE_NOMINAL_SUBJECTS = True

USE_ADVERBS = True
TENSES = ["present_perfect"]

RANDOM_SUBJECT_SELECTION = True
RANDOM_OUTPUT_SAMPLE = True
REMOVE_DUPLICATE_SENTENCES = True

N_RANDOM_ROWS = 10
RANDOM_STATE = 99

SPLIT_OUTPUT = False
ROWS_PER_FILE = 10000

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_ES_v1_utf8.csv"
file2 = "sentence_buildersES_utf8.csv"
file3 = "spanish_verbs_inflection_sub_utf8.csv"

OUTPUT_DIR = "nominal_subj"
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_file = os.path.join(
    OUTPUT_DIR,
    "LVC_ES_active_subjects_output.csv"
)

random.seed(RANDOM_STATE)

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1, sep=None, engine="python", encoding="utf8")
builder = pd.read_csv(file2, sep=None, engine="python", encoding="utf8")
verbs = pd.read_csv(file3, sep=None, engine="python", encoding="utf8")

for df in [items, builder, verbs]:
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.replace("\n", "", regex=False)
        .str.replace("\r", "", regex=False)
    )
    df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], inplace=True)
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

# =========================
# HELPERS
# =========================

def clean_list(series):
    values = series.astype(str).str.strip().replace("nan", pd.NA).dropna()
    values = values[values != ""]
    return values.unique().tolist()

def get_col(row, possible_names):
    for name in possible_names:
        if name in row.index:
            value = str(row.get(name, "")).strip()
            if value != "nan":
                return value
    return ""

def make_np(det, noun):
    if det in ["", "NA"]:
        return noun
    return f"{det} {noun}"

def get_object_phrase(item):
    """
    Prefer full_expression because it may contain articles.
    Fall back to object_es.
    """
    verb = str(item["verb_es"]).strip()
    full_expression = str(item.get("full_expression", "")).strip()
    object_es = str(item.get("object_es", "")).strip()

    if full_expression and full_expression != "nan":
        if full_expression.startswith(verb + " "):
            return full_expression.replace(verb + " ", "", 1)
        parts = full_expression.split(" ", 1)
        if len(parts) == 2:
            return parts[1]

    return object_es

def get_verb_form(verb, person, tense_col):
    match = verbs[
        (verbs["verb"] == verb) &
        (verbs["person"] == person)
    ]

    if match.empty:
        raise ValueError(
            f"No verb form for verb={verb}, person={person}, tense={tense_col}"
        )

    return match.iloc[0][tense_col]

def add_adverb(verb_form, adverb, tense):
    if adverb == "NA":
        return verb_form

    if tense == "future":
        return f"{verb_form} {adverb}"

    aux, participle = verb_form.split(" ", 1)
    return f"{aux} {adverb} {participle}"

# =========================
# BASIC LISTS
# =========================

time_specs = clean_list(builder["time_specs"])
adverbs = clean_list(builder["adverbs"])

# Separate singular determiner columns
det_sg_masc = clean_list(builder["det_sg_masc"])
det_sg_fem = clean_list(builder["det_sg_fem"])

# Separate plural determiner columns
det_pl_fem = clean_list(builder["det_pl_fem"])
det_pl_masc = clean_list(builder["det_pl_masc"])

poss_masc = clean_list(builder["poss_masc"])
poss_fem = clean_list(builder["poss_fem"])
poss_pl = clean_list(builder["poss_pl"])

pronouns = clean_list(builder["subjects_pron"])

# =========================
# SUBJECT PHRASES
# =========================

subject_rows = []

# Pronoun subjects
if USE_PRONOUN_SUBJECTS:
    for pron in pronouns:

        subject_rows.append({
            "subject": pron,
            "subject_base": pron,
            "subject_number": (
                "sg"
                if pron in ["yo", "tú", "él", "ella"]
                else "pl"
            ),
            "subject_det": "NA",
            "subject_det_type": "NA",
            "subject_type": "pronoun",
            "person": pron,
            "context_gender": "NA",
            "possessive": "NA",
            "subject_realized": "no"
        })

# =========================
# NOMINAL SUBJECTS
# =========================

if USE_NOMINAL_SUBJECTS:

    for _, row in builder.iterrows():

        subj_sg = get_col(row, ["subject_sg"])
        subj_pl = get_col(row, ["subject_pl"])

        context_gender_sg = get_col(
            row,
            ["context_gender_sg"]
        )

        context_gender_pl = get_col(
            row,
            ["context_gender_pl"]
        )

        possessive = get_col(
            row,
            ["det_poss_tf"]
        )

        # -------------------------
        # Singular nominal subjects
        # -------------------------

        if subj_sg not in ["", "nan"]:

            if context_gender_sg == "F":
                person = "ella"
                dets_sg = det_sg_fem
                poss_dets_sg = poss_fem
            else:
                person = "él"
                dets_sg = det_sg_masc
                poss_dets_sg = poss_masc

            # Ordinary singular determiners
            for det in dets_sg:

                subject_rows.append({
                    "subject": make_np(det, subj_sg),
                    "subject_base": subj_sg,
                    "subject_number": "sg",
                    "subject_det": det,
                    "subject_det_type": "det",
                    "subject_type": "nominal",
                    "person": person,
                    "context_gender": context_gender_sg,
                    "possessive": possessive,
                    "subject_realized": "yes"
                })

            # Singular possessive determiners
            if possessive == "T":

                for det in poss_dets_sg:

                    subject_rows.append({
                        "subject": make_np(det, subj_sg),
                        "subject_base": subj_sg,
                        "subject_number": "sg",
                        "subject_det": det,
                        "subject_det_type": "det_poss",
                        "subject_type": "nominal",
                        "person": person,
                        "context_gender": context_gender_sg,
                        "possessive": possessive,
                        "subject_realized": "yes"
                    })

        # -------------------------
        # Plural nominal subjects
        # -------------------------

        if subj_pl not in ["", "nan"]:

            # Select plural person and determiner list
            # directly from the grammatical gender.
            if context_gender_pl == "F":
                person = "ellas"
                dets_pl = det_pl_fem
            else:
                person = "ellos"
                dets_pl = det_pl_masc

            # No determiner
            subject_rows.append({
                "subject": subj_pl,
                "subject_base": subj_pl,
                "subject_number": "pl",
                "subject_det": "NA",
                "subject_det_type": "no_det",
                "subject_type": "nominal",
                "person": person,
                "context_gender": context_gender_pl,
                "possessive": possessive,
                "subject_realized": "yes"
            })

            # Ordinary plural determiners
            for det in dets_pl:

                subject_rows.append({
                    "subject": make_np(det, subj_pl),
                    "subject_base": subj_pl,
                    "subject_number": "pl",
                    "subject_det": det,
                    "subject_det_type": "det",
                    "subject_type": "nominal",
                    "person": person,
                    "context_gender": context_gender_pl,
                    "possessive": possessive,
                    "subject_realized": "yes"
                })

            # Plural possessive determiners
            if possessive == "T":

                for det in poss_pl:

                    subject_rows.append({
                        "subject": make_np(det, subj_pl),
                        "subject_base": subj_pl,
                        "subject_number": "pl",
                        "subject_det": det,
                        "subject_det_type": "det_poss",
                        "subject_type": "nominal",
                        "person": person,
                        "context_gender": context_gender_pl,
                        "possessive": possessive,
                        "subject_realized": "yes"
                    })

subjects_df = pd.DataFrame(
    subject_rows
).drop_duplicates()
# =========================
# FILTER SUBJECTS
# =========================

if SUBJECT_NUMBER != "both":
    subjects_df = subjects_df[subjects_df["subject_number"] == SUBJECT_NUMBER]

nominal_subjects = subjects_df[
    (subjects_df["subject_type"] == "nominal") &
    (subjects_df["subject_det_type"].isin(SUBJECT_DET_TYPES))
]

if USE_PRONOUN_SUBJECTS:
    pronoun_subjects = subjects_df[subjects_df["subject_type"] == "pronoun"]
    subjects_df = pd.concat([pronoun_subjects, nominal_subjects], ignore_index=True)
else:
    subjects_df = nominal_subjects

subjects_df = subjects_df.reset_index(drop=True)

print("Number of available subjects:", len(subjects_df))
print(subjects_df["subject"].head(50))
print(subjects_df["subject_type"].value_counts())

subject_records = subjects_df.to_dict("records")

# =========================
# TENSE COLUMNS
# =========================

tense_columns = {
    "future": "future_active",
    "present_perfect": "present_perfect_active",
    "past_perfect": "past_perfect_active",
}

# =========================
# BUILD SENTENCES
# =========================

rows = []

for _, item in items.iterrows():

    verb = item["verb_es"]
    obj = get_object_phrase(item)

    for time_spec in time_specs:

        adverb_options = adverbs if USE_ADVERBS else ["NA"]

        for adverb in adverb_options:
            for tense in TENSES:

                tense_col = tense_columns[tense]

                if RANDOM_SUBJECT_SELECTION:
                    subject_iterator = [random.choice(subject_records)]
                else:
                    subject_iterator = subject_records

                for subj_info in subject_iterator:

                    subject = subj_info["subject"]
                    person = subj_info["person"]

                    verb_form = get_verb_form(verb, person, tense_col)
                    verb_form = add_adverb(verb_form, adverb, tense)

                    if subj_info["subject_type"] == "pronoun":
                        sentence_body = f"{verb_form} {obj}"
                    else:
                        sentence_body = f"{subject} {verb_form} {obj}"

                    sentence = f"{time_spec}, {sentence_body}."

                    new_row = item.to_dict()
                    new_row.update(subj_info)
                    new_row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "time_spec": time_spec
                    })

                    rows.append(new_row)

# =========================
# FINAL OUTPUT
# =========================

out = pd.DataFrame(rows)

if REMOVE_DUPLICATE_SENTENCES:
    out = out.drop_duplicates(subset=["sentence"])

if RANDOM_OUTPUT_SAMPLE:
    out = out.sample(
        n=min(N_RANDOM_ROWS, len(out)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

out = out.dropna(axis=1, how="all")

empty_cols = []
for col in out.columns:
    values = out[col].astype(str).str.strip().replace("nan", "")
    if (values == "").all():
        empty_cols.append(col)

out = out.drop(columns=empty_cols)

# =========================
# SAVE
# =========================

if not SPLIT_OUTPUT:

    out.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Saved {len(out)} rows to {output_file}")

else:

    n_files = (len(out) - 1) // ROWS_PER_FILE + 1

    for i in range(n_files):

        start = i * ROWS_PER_FILE
        end = min((i + 1) * ROWS_PER_FILE, len(out))

        chunk = out.iloc[start:end]

        filename = output_file.replace(
            ".csv",
            f"_part{i+1:03d}.csv"
        )

        chunk.to_csv(filename, index=False, encoding="utf-8-sig")

        print(f"Saved rows {start+1:,}-{end:,} to {filename}")

    print(f"\nFinished: {len(out):,} rows written to {n_files} files.")

In [ ]:
import pandas as pd
import random
import os

# =========================
# SETTINGS
# =========================

SUBJECT_NUMBER = "both"      # "sg", "pl", "both"
SUBJECT_DET_TYPES = ["det"]  # "det", "det_poss", "no_det"

USE_PRONOUN_SUBJECTS = False
USE_NOMINAL_SUBJECTS = True

USE_ADVERBS = True
TENSES = ["present_perfect"]

RANDOM_SUBJECT_SELECTION = True
RANDOM_OUTPUT_SAMPLE = True
REMOVE_DUPLICATE_SENTENCES = True

N_RANDOM_ROWS = 10
RANDOM_STATE = 99

SPLIT_OUTPUT = False
ROWS_PER_FILE = 10000

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_ES_v1_utf8.csv"
file2 = "sentence_buildersES_utf8.csv"
file3 = "spanish_verbs_inflection_sub_utf8.csv"

OUTPUT_DIR = "nominal_subj"
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_file = os.path.join(
    OUTPUT_DIR,
    "LVC_ES_active_subjects_output.csv"
)

random.seed(RANDOM_STATE)

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1, sep=None, engine="python", encoding="utf8")
builder = pd.read_csv(file2, sep=None, engine="python", encoding="utf8")
verbs = pd.read_csv(file3, sep=None, engine="python", encoding="utf8")

for df in [items, builder, verbs]:
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.replace("\n", "", regex=False)
        .str.replace("\r", "", regex=False)
    )
    df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], inplace=True)
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

# =========================
# HELPERS
# =========================

def clean_list(series):
    values = series.astype(str).str.strip().replace("nan", pd.NA).dropna()
    values = values[values != ""]
    return values.unique().tolist()

def get_col(row, possible_names):
    for name in possible_names:
        if name in row.index:
            value = str(row.get(name, "")).strip()
            if value != "nan":
                return value
    return ""

def make_np(det, noun):
    if det in ["", "NA"]:
        return noun
    return f"{det} {noun}"

# def plural_dets_for_gender(dets, context_gender):
#     """
#     Spanish needs masculine/feminine plural determiners.
#     The builder has one plural column, so we adapt los/unos to las/unas when needed.
#     """
#     output = []

#     for det in dets:
#         if context_gender == "F":
#             if det == "los":
#                 output.append("las")
#             elif det == "unos":
#                 output.append("unas")
#             else:
#                 output.append(det)
#         else:
#             output.append(det)

#     return output

def get_object_phrase(item):
    """
    Prefer full_expression because it may contain articles.
    Fall back to object_es.
    """
    verb = str(item["verb_es"]).strip()
    full_expression = str(item.get("full_expression", "")).strip()
    object_es = str(item.get("object_es", "")).strip()

    if full_expression and full_expression != "nan":
        if full_expression.startswith(verb + " "):
            return full_expression.replace(verb + " ", "", 1)
        parts = full_expression.split(" ", 1)
        if len(parts) == 2:
            return parts[1]

    return object_es

def get_verb_form(verb, person, tense_col):
    match = verbs[
        (verbs["verb"] == verb) &
        (verbs["person"] == person)
    ]

    if match.empty:
        raise ValueError(
            f"No verb form for verb={verb}, person={person}, tense={tense_col}"
        )

    return match.iloc[0][tense_col]

def add_adverb(verb_form, adverb, tense):
    if adverb == "NA":
        return verb_form

    if tense == "future":
        return f"{verb_form} {adverb}"

    aux, participle = verb_form.split(" ", 1)
    return f"{aux} {adverb} {participle}"

# =========================
# BASIC LISTS
# =========================

time_specs = clean_list(builder["time_specs"])
adverbs = clean_list(builder["adverbs"])

det_masc = clean_list(builder["det_masc"])
det_fem = clean_list(builder["det_fem"])

# New separate plural determiner columns
det_pl_fem = clean_list(builder["det_pl_fem"])
det_pl_masc = clean_list(builder["det_pl_masc"])

poss_masc = clean_list(builder["poss_masc"])
poss_fem = clean_list(builder["poss_fem"])
poss_pl = clean_list(builder["poss_pl"])

pronouns = clean_list(builder["subjects_pron"])

# =========================
# SUBJECT PHRASES
# =========================

subject_rows = []

# Pronoun subjects
if USE_PRONOUN_SUBJECTS:
    for pron in pronouns:

        subject_rows.append({
            "subject": pron,
            "subject_base": pron,
            "subject_number": (
                "sg"
                if pron in ["yo", "tú", "él", "ella"]
                else "pl"
            ),
            "subject_det": "NA",
            "subject_det_type": "NA",
            "subject_type": "pronoun",
            "person": pron,
            "context_gender": "NA",
            "possessive": "NA",
            "subject_realized": "no"
        })

# =========================
# NOMINAL SUBJECTS
# =========================

if USE_NOMINAL_SUBJECTS:

    for _, row in builder.iterrows():

        subj_sg = get_col(row, ["subject_sg"])
        subj_pl = get_col(row, ["subject_pl"])

        context_gender_sg = get_col(
            row,
            ["context_gender_sg"]
        )

        context_gender_pl = get_col(
            row,
            ["context_gender_pl"]
        )

        possessive = get_col(
            row,
            ["det_poss_tf"]
        )

        # -------------------------
        # Singular nominal subjects
        # -------------------------

        if subj_sg not in ["", "nan"]:

            if context_gender_sg == "F":
                person = "ella"
                dets_sg = det_fem
                poss_dets_sg = poss_fem
            else:
                person = "él"
                dets_sg = det_masc
                poss_dets_sg = poss_masc

            # Ordinary singular determiners
            for det in dets_sg:

                subject_rows.append({
                    "subject": make_np(det, subj_sg),
                    "subject_base": subj_sg,
                    "subject_number": "sg",
                    "subject_det": det,
                    "subject_det_type": "det",
                    "subject_type": "nominal",
                    "person": person,
                    "context_gender": context_gender_sg,
                    "possessive": possessive,
                    "subject_realized": "yes"
                })

            # Singular possessive determiners
            if possessive == "T":

                for det in poss_dets_sg:

                    subject_rows.append({
                        "subject": make_np(det, subj_sg),
                        "subject_base": subj_sg,
                        "subject_number": "sg",
                        "subject_det": det,
                        "subject_det_type": "det_poss",
                        "subject_type": "nominal",
                        "person": person,
                        "context_gender": context_gender_sg,
                        "possessive": possessive,
                        "subject_realized": "yes"
                    })

        # -------------------------
        # Plural nominal subjects
        # -------------------------

        if subj_pl not in ["", "nan"]:

            # Select plural person and determiner list
            # directly from the grammatical gender.
            if context_gender_pl == "F":
                person = "ellas"
                dets_pl = det_pl_fem
            else:
                person = "ellos"
                dets_pl = det_pl_masc

            # No determiner
            subject_rows.append({
                "subject": subj_pl,
                "subject_base": subj_pl,
                "subject_number": "pl",
                "subject_det": "NA",
                "subject_det_type": "no_det",
                "subject_type": "nominal",
                "person": person,
                "context_gender": context_gender_pl,
                "possessive": possessive,
                "subject_realized": "yes"
            })

            # Ordinary plural determiners
            for det in dets_pl:

                subject_rows.append({
                    "subject": make_np(det, subj_pl),
                    "subject_base": subj_pl,
                    "subject_number": "pl",
                    "subject_det": det,
                    "subject_det_type": "det",
                    "subject_type": "nominal",
                    "person": person,
                    "context_gender": context_gender_pl,
                    "possessive": possessive,
                    "subject_realized": "yes"
                })

            # Plural possessive determiners
            if possessive == "T":

                for det in poss_pl:

                    subject_rows.append({
                        "subject": make_np(det, subj_pl),
                        "subject_base": subj_pl,
                        "subject_number": "pl",
                        "subject_det": det,
                        "subject_det_type": "det_poss",
                        "subject_type": "nominal",
                        "person": person,
                        "context_gender": context_gender_pl,
                        "possessive": possessive,
                        "subject_realized": "yes"
                    })

subjects_df = pd.DataFrame(
    subject_rows
).drop_duplicates()
# =========================
# FILTER SUBJECTS
# =========================

if SUBJECT_NUMBER != "both":
    subjects_df = subjects_df[subjects_df["subject_number"] == SUBJECT_NUMBER]

nominal_subjects = subjects_df[
    (subjects_df["subject_type"] == "nominal") &
    (subjects_df["subject_det_type"].isin(SUBJECT_DET_TYPES))
]

if USE_PRONOUN_SUBJECTS:
    pronoun_subjects = subjects_df[subjects_df["subject_type"] == "pronoun"]
    subjects_df = pd.concat([pronoun_subjects, nominal_subjects], ignore_index=True)
else:
    subjects_df = nominal_subjects

subjects_df = subjects_df.reset_index(drop=True)

print("Number of available subjects:", len(subjects_df))
print(subjects_df["subject"].head(50))
print(subjects_df["subject_type"].value_counts())

subject_records = subjects_df.to_dict("records")

# =========================
# TENSE COLUMNS
# =========================

tense_columns = {
    "future": "future_active",
    "present_perfect": "present_perfect_active",
    "past_perfect": "past_perfect_active",
}

# =========================
# BUILD SENTENCES
# =========================

rows = []

for _, item in items.iterrows():

    verb = item["verb_es"]
    obj = get_object_phrase(item)

    for time_spec in time_specs:

        adverb_options = adverbs if USE_ADVERBS else ["NA"]

        for adverb in adverb_options:
            for tense in TENSES:

                tense_col = tense_columns[tense]

                if RANDOM_SUBJECT_SELECTION:
                    subject_iterator = [random.choice(subject_records)]
                else:
                    subject_iterator = subject_records

                for subj_info in subject_iterator:

                    subject = subj_info["subject"]
                    person = subj_info["person"]

                    verb_form = get_verb_form(verb, person, tense_col)
                    verb_form = add_adverb(verb_form, adverb, tense)

                    if subj_info["subject_type"] == "pronoun":
                        sentence_body = f"{verb_form} {obj}"
                    else:
                        sentence_body = f"{subject} {verb_form} {obj}"

                    sentence = f"{time_spec}, {sentence_body}."

                    new_row = item.to_dict()
                    new_row.update(subj_info)
                    new_row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "time_spec": time_spec
                    })

                    rows.append(new_row)

# =========================
# FINAL OUTPUT
# =========================

out = pd.DataFrame(rows)

if REMOVE_DUPLICATE_SENTENCES:
    out = out.drop_duplicates(subset=["sentence"])

if RANDOM_OUTPUT_SAMPLE:
    out = out.sample(
        n=min(N_RANDOM_ROWS, len(out)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

out = out.dropna(axis=1, how="all")

empty_cols = []
for col in out.columns:
    values = out[col].astype(str).str.strip().replace("nan", "")
    if (values == "").all():
        empty_cols.append(col)

out = out.drop(columns=empty_cols)

# =========================
# SAVE
# =========================

if not SPLIT_OUTPUT:

    out.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Saved {len(out)} rows to {output_file}")

else:

    n_files = (len(out) - 1) // ROWS_PER_FILE + 1

    for i in range(n_files):

        start = i * ROWS_PER_FILE
        end = min((i + 1) * ROWS_PER_FILE, len(out))

        chunk = out.iloc[start:end]

        filename = output_file.replace(
            ".csv",
            f"_part{i+1:03d}.csv"
        )

        chunk.to_csv(filename, index=False, encoding="utf-8-sig")

        print(f"Saved rows {start+1:,}-{end:,} to {filename}")

    print(f"\nFinished: {len(out):,} rows written to {n_files} files.")

In [ ]:
import pandas as pd
import random
import os

# =========================
# SETTINGS
# =========================

SUBJECT_NUMBER = "both"      # "sg", "pl", "both"
SUBJECT_DET_TYPES = ["det"]  # "det", "det_poss", "no_det"

USE_PRONOUN_SUBJECTS = False
USE_NOMINAL_SUBJECTS = True

USE_ADVERBS = True
TENSES = ["present_perfect"]

RANDOM_SUBJECT_SELECTION = True
RANDOM_OUTPUT_SAMPLE = True
REMOVE_DUPLICATE_SENTENCES = True

N_RANDOM_ROWS = 10
RANDOM_STATE = 99

SPLIT_OUTPUT = False
ROWS_PER_FILE = 10000

# =========================
# FILE PATHS
# =========================

file1 = "collocation_items_ES_v1_utf8.csv"
file2 = "sentence_buildersES_utf8.csv"
file3 = "spanish_verbs_inflection_sub_utf8.csv"

OUTPUT_DIR = "nominal_subj"
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_file = os.path.join(
    OUTPUT_DIR,
    "LVC_ES_active_subjects_output.csv"
)

random.seed(RANDOM_STATE)

# =========================
# READ FILES
# =========================

items = pd.read_csv(file1, sep=None, engine="python", encoding="utf8")
builder = pd.read_csv(file2, sep=None, engine="python", encoding="utf8")
verbs = pd.read_csv(file3, sep=None, engine="python", encoding="utf8")

for df in [items, builder, verbs]:
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.replace("\n", "", regex=False)
        .str.replace("\r", "", regex=False)
    )
    df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], inplace=True)
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

# =========================
# HELPERS
# =========================

def clean_list(series):
    values = series.astype(str).str.strip().replace("nan", pd.NA).dropna()
    values = values[values != ""]
    return values.unique().tolist()

def get_col(row, possible_names):
    for name in possible_names:
        if name in row.index:
            value = str(row.get(name, "")).strip()
            if value != "nan":
                return value
    return ""

def make_np(det, noun):
    if det in ["", "NA"]:
        return noun
    return f"{det} {noun}"

# def plural_dets_for_gender(dets, context_gender):
#     """
#     Spanish needs masculine/feminine plural determiners.
#     The builder has one plural column, so we adapt los/unos to las/unas when needed.
#     """
#     output = []

#     for det in dets:
#         if context_gender == "F":
#             if det == "los":
#                 output.append("las")
#             elif det == "unos":
#                 output.append("unas")
#             else:
#                 output.append(det)
#         else:
#             output.append(det)

#     return output

def get_object_phrase(item):
    """
    Prefer full_expression because it may contain articles.
    Fall back to object_es.
    """
    verb = str(item["verb_es"]).strip()
    full_expression = str(item.get("full_expression", "")).strip()
    object_es = str(item.get("object_es", "")).strip()

    if full_expression and full_expression != "nan":
        if full_expression.startswith(verb + " "):
            return full_expression.replace(verb + " ", "", 1)
        parts = full_expression.split(" ", 1)
        if len(parts) == 2:
            return parts[1]

    return object_es

def get_verb_form(verb, person, tense_col):
    match = verbs[
        (verbs["verb"] == verb) &
        (verbs["person"] == person)
    ]

    if match.empty:
        raise ValueError(
            f"No verb form for verb={verb}, person={person}, tense={tense_col}"
        )

    return match.iloc[0][tense_col]

def add_adverb(verb_form, adverb, tense):
    if adverb == "NA":
        return verb_form

    if tense == "future":
        return f"{verb_form} {adverb}"

    aux, participle = verb_form.split(" ", 1)
    return f"{aux} {adverb} {participle}"

# =========================
# BASIC LISTS
# =========================

time_specs = clean_list(builder["time_specs"])
adverbs = clean_list(builder["adverbs"])

det_masc = clean_list(builder["det_masc"])
det_fem = clean_list(builder["det_fem"])

# New separate plural determiner columns
det_pl_fem = clean_list(builder["det_pl_fem"])
det_pl_masc = clean_list(builder["det_pl_masc"])

poss_masc = clean_list(builder["poss_masc"])
poss_fem = clean_list(builder["poss_fem"])
poss_pl = clean_list(builder["poss_pl"])

pronouns = clean_list(builder["subjects_pron"])

# =========================
# SUBJECT PHRASES
# =========================

subject_rows = []

# Pronoun subjects
if USE_PRONOUN_SUBJECTS:
    for pron in pronouns:

        subject_rows.append({
            "subject": pron,
            "subject_base": pron,
            "subject_number": (
                "sg"
                if pron in ["yo", "tú", "él", "ella"]
                else "pl"
            ),
            "subject_det": "NA",
            "subject_det_type": "NA",
            "subject_type": "pronoun",
            "person": pron,
            "context_gender": "NA",
            "possessive": "NA",
            "subject_realized": "no"
        })

# =========================
# NOMINAL SUBJECTS
# =========================

if USE_NOMINAL_SUBJECTS:

    for _, row in builder.iterrows():

        subj_sg = get_col(row, ["subject_sg"])
        subj_pl = get_col(row, ["subject_pl"])

        context_gender_sg = get_col(
            row,
            ["context_gender_sg"]
        )

        context_gender_pl = get_col(
            row,
            ["context_gender_pl"]
        )

        possessive = get_col(
            row,
            ["det_poss_tf"]
        )

        # -------------------------
        # Singular nominal subjects
        # -------------------------

        if subj_sg not in ["", "nan"]:

            if context_gender_sg == "F":
                person = "ella"
                dets_sg = det_fem
                poss_dets_sg = poss_fem
            else:
                person = "él"
                dets_sg = det_masc
                poss_dets_sg = poss_masc

            # Ordinary singular determiners
            for det in dets_sg:

                subject_rows.append({
                    "subject": make_np(det, subj_sg),
                    "subject_base": subj_sg,
                    "subject_number": "sg",
                    "subject_det": det,
                    "subject_det_type": "det",
                    "subject_type": "nominal",
                    "person": person,
                    "context_gender": context_gender_sg,
                    "possessive": possessive,
                    "subject_realized": "yes"
                })

            # Singular possessive determiners
            if possessive == "T":

                for det in poss_dets_sg:

                    subject_rows.append({
                        "subject": make_np(det, subj_sg),
                        "subject_base": subj_sg,
                        "subject_number": "sg",
                        "subject_det": det,
                        "subject_det_type": "det_poss",
                        "subject_type": "nominal",
                        "person": person,
                        "context_gender": context_gender_sg,
                        "possessive": possessive,
                        "subject_realized": "yes"
                    })

        # -------------------------
        # Plural nominal subjects
        # -------------------------

        if subj_pl not in ["", "nan"]:

            # Select plural person and determiner list
            # directly from the grammatical gender.
            if context_gender_pl == "F":
                person = "ellas"
                dets_pl = det_pl_fem
            else:
                person = "ellos"
                dets_pl = det_pl_masc

            # No determiner
            subject_rows.append({
                "subject": subj_pl,
                "subject_base": subj_pl,
                "subject_number": "pl",
                "subject_det": "NA",
                "subject_det_type": "no_det",
                "subject_type": "nominal",
                "person": person,
                "context_gender": context_gender_pl,
                "possessive": possessive,
                "subject_realized": "yes"
            })

            # Ordinary plural determiners
            for det in dets_pl:

                subject_rows.append({
                    "subject": make_np(det, subj_pl),
                    "subject_base": subj_pl,
                    "subject_number": "pl",
                    "subject_det": det,
                    "subject_det_type": "det",
                    "subject_type": "nominal",
                    "person": person,
                    "context_gender": context_gender_pl,
                    "possessive": possessive,
                    "subject_realized": "yes"
                })

            # Plural possessive determiners
            if possessive == "T":

                for det in poss_pl:

                    subject_rows.append({
                        "subject": make_np(det, subj_pl),
                        "subject_base": subj_pl,
                        "subject_number": "pl",
                        "subject_det": det,
                        "subject_det_type": "det_poss",
                        "subject_type": "nominal",
                        "person": person,
                        "context_gender": context_gender_pl,
                        "possessive": possessive,
                        "subject_realized": "yes"
                    })

subjects_df = pd.DataFrame(
    subject_rows
).drop_duplicates()
# =========================
# FILTER SUBJECTS
# =========================

if SUBJECT_NUMBER != "both":
    subjects_df = subjects_df[subjects_df["subject_number"] == SUBJECT_NUMBER]

nominal_subjects = subjects_df[
    (subjects_df["subject_type"] == "nominal") &
    (subjects_df["subject_det_type"].isin(SUBJECT_DET_TYPES))
]

if USE_PRONOUN_SUBJECTS:
    pronoun_subjects = subjects_df[subjects_df["subject_type"] == "pronoun"]
    subjects_df = pd.concat([pronoun_subjects, nominal_subjects], ignore_index=True)
else:
    subjects_df = nominal_subjects

subjects_df = subjects_df.reset_index(drop=True)

print("Number of available subjects:", len(subjects_df))
print(subjects_df["subject"].head(50))
print(subjects_df["subject_type"].value_counts())

subject_records = subjects_df.to_dict("records")

# =========================
# TENSE COLUMNS
# =========================

tense_columns = {
    "future": "future_active",
    "present_perfect": "present_perfect_active",
    "past_perfect": "past_perfect_active",
}

# =========================
# BUILD SENTENCES
# =========================

rows = []

for _, item in items.iterrows():

    verb = item["verb_es"]
    obj = get_object_phrase(item)

    for time_spec in time_specs:

        adverb_options = adverbs if USE_ADVERBS else ["NA"]

        for adverb in adverb_options:
            for tense in TENSES:

                tense_col = tense_columns[tense]

                if RANDOM_SUBJECT_SELECTION:
                    subject_iterator = [random.choice(subject_records)]
                else:
                    subject_iterator = subject_records

                for subj_info in subject_iterator:

                    subject = subj_info["subject"]
                    person = subj_info["person"]

                    verb_form = get_verb_form(verb, person, tense_col)
                    verb_form = add_adverb(verb_form, adverb, tense)

                    if subj_info["subject_type"] == "pronoun":
                        sentence_body = f"{verb_form} {obj}"
                    else:
                        sentence_body = f"{subject} {verb_form} {obj}"

                    sentence = f"{time_spec}, {sentence_body}."

                    new_row = item.to_dict()
                    new_row.update(subj_info)
                    new_row.update({
                        "sentence": sentence,
                        "voice": "active",
                        "tense": tense,
                        "adverb": adverb,
                        "time_spec": time_spec
                    })

                    rows.append(new_row)

# =========================
# FINAL OUTPUT
# =========================

out = pd.DataFrame(rows)

if REMOVE_DUPLICATE_SENTENCES:
    out = out.drop_duplicates(subset=["sentence"])

if RANDOM_OUTPUT_SAMPLE:
    out = out.sample(
        n=min(N_RANDOM_ROWS, len(out)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

out = out.dropna(axis=1, how="all")

empty_cols = []
for col in out.columns:
    values = out[col].astype(str).str.strip().replace("nan", "")
    if (values == "").all():
        empty_cols.append(col)

out = out.drop(columns=empty_cols)

# =========================
# SAVE
# =========================

if not SPLIT_OUTPUT:

    out.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Saved {len(out)} rows to {output_file}")

else:

    n_files = (len(out) - 1) // ROWS_PER_FILE + 1

    for i in range(n_files):

        start = i * ROWS_PER_FILE
        end = min((i + 1) * ROWS_PER_FILE, len(out))

        chunk = out.iloc[start:end]

        filename = output_file.replace(
            ".csv",
            f"_part{i+1:03d}.csv"
        )

        chunk.to_csv(filename, index=False, encoding="utf-8-sig")

        print(f"Saved rows {start+1:,}-{end:,} to {filename}")

    print(f"\nFinished: {len(out):,} rows written to {n_files} files.")